## Download the tables. These tables are already ingested in:
- CATALOG: energy_endava_193
- SCHEMA: default


In [0]:

from pyspark.sql.functions import year, month, dayofmonth, col, to_date, lit

# Read base tables from Delta Lake
nsw_dispatch_5min = spark.table("energy_endava_193.default.nsw_dispatch_5min")
nsw_demand = spark.table("energy_endava_193.default.nsw_demand")
nsw_scada = spark.table("energy_endava_193.default.nsw_scada")
nsw_dictionary_mapped = spark.table("energy_endava_193.default.nsw_dictionary_mapped")
nsw_dispatch_demand = spark.table("energy_endava_193.default.nsw_dispatch_demand")
nsw_dispatch_demand_peak_2022 = spark.table("energy_endava_193.default.nsw_dispatch_demand_peak_2022")
nsw_scada_peak_2022 = spark.table("energy_endava_193.default.nsw_scada_peak_2022")
nsw_dictionary_peak_2022 = spark.table("energy_endava_193.default.nsw_dictionary_peak_2022")

In [0]:
display(nsw_dictionary_peak_2022)

In [0]:
display(nsw_scada_peak_2022)

In [0]:
from pyspark.sql.functions import broadcast

# Deduplicate dictionary by DUID (keeps first occurrence)
# This reduces dictionary from 1,943 to 625 unique DUIDs
nsw_dictionary_peak_2022 = nsw_dictionary_peak_2022.dropDuplicates(["DUID"])

print(f"📊 Dictionary deduplicated: {nsw_dictionary_peak_2022.count():,} unique DUIDs")

# Broadcast join with deduplicated dictionary
# This avoids creating 272,920 unnecessary duplicate rows
nsw_scada_with_fuel = nsw_scada_peak_2022.join(
    broadcast(nsw_dictionary_peak_2022.select("DUID", "FUELSOURCEPRIMARY", "DISPATCHTYPE", "REGIONID", "STARTTYPE", "SCHEDULE_TYPE")),
    on="DUID",
    how="left"
)

print(f"✅ Joined {nsw_scada_with_fuel.count():,} rows (1:1 mapping)")
print(f"\nColumns: {nsw_scada_with_fuel.columns}")
print("\nSample data:")
display(nsw_scada_with_fuel.limit(10))

In [0]:
from pyspark.sql.functions import when

print("="*70)
print("Creating Generators-Only Dataset")
print("="*70)

# Fix fuel sources for generators with missing data
# PORTWF: Wind farm
# PIONEER: Pioneer Sugar Mill cogeneration using bagasse (sugar cane waste)
nsw_scada_generators = nsw_scada_with_fuel.withColumn(
    "FUELSOURCEPRIMARY",
    when(col("DUID") == "PORTWF", "Wind")
    .when(col("DUID") == "PIONEER", "Renewable/ Biomass / Waste")
    .otherwise(col("FUELSOURCEPRIMARY"))
)

# Filter to only include generators (exclude LOAD units)
nsw_scada_generators = nsw_scada_generators.filter(col("DISPATCHTYPE") == "GENERATOR")

print(f"\nTotal generator rows: {nsw_scada_generators.count():,}")

# Check fuel source distribution for generators only
print("\n" + "="*70)
print("Fuel Source Distribution (Generators Only)")
print("="*70)

generators_fuel_dist = nsw_scada_generators.groupBy("FUELSOURCEPRIMARY") \
    .count() \
    .orderBy(col("count").desc())

print(f"\nUnique fuel sources: {generators_fuel_dist.count()}")
display(generators_fuel_dist)

# Check for remaining null values
null_count = nsw_scada_generators.filter(col("FUELSOURCEPRIMARY").isNull()).count()
print(f"\n✅ Remaining NULL values: {null_count}")
print("\n✅ Dataset ready: nsw_scada_generators (generators only, all fuel sources fixed)")

print("\n" + "="*70)
print("Dataset (nsw_scada_generators):")
print("="*70)
display(nsw_scada_generators)

In [0]:
print("="*70)
print("Fuel Source & Start Type Distribution")
print("="*70)

# Count unique pairs of FUELSOURCEPRIMARY and STARTTYPE
fuel_starttype_dist = nsw_scada_generators.groupBy("FUELSOURCEPRIMARY", "STARTTYPE", "SCHEDULE_TYPE") \
    .count() \
    .orderBy([col("FUELSOURCEPRIMARY"), col("count").desc()])

print(f"\nTotal rows: {nsw_scada_generators.count():,}")
print(f"Unique (FUELSOURCEPRIMARY, STARTTYPE) pairs: {fuel_starttype_dist.count()}\n")

print("Distribution by Fuel Source and Start Type:")
display(fuel_starttype_dist)

In [0]:
display(nsw_dispatch_demand_peak_2022)

In [0]:
display(nsw_dictionary_mapped)

In [0]:
display(nsw_scada_generators)